# [SETUP](https://makkoen.github.io/logprobs-workshop/before-workshop/)

In [ ]:
import os
from google.colab import userdata

# Ensure your OpenAI API key is saved as a secret in Colab
# under the name 'OPENAI_API_KEY'.
# Click the key icon on the left sidebar to manage secrets.
openai_api_key = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = openai_api_key

print('OPENAI_API_KEY loaded successfully from Colab Secrets.')

In [ ]:
!pip install optuna
!pip install shap
!pip install -U gdown

import gdown
import json
import math
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import optuna.visualization as vis
import shap
from tqdm import tqdm
from openai import OpenAI
from sklearn.metrics import precision_recall_curve, auc, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_similarity

# Initialize OpenAI client
client = OpenAI()

# Download the datasets
filtering_url = 'https://drive.google.com/file/d/16He3YIPwqexOMRYiEoW_df9t5mEQID-Q/view?usp=drive_link'
output_filename = 'filtering_dataset.csv'
gdown.download(filtering_url, output_filename, quiet=False)
print(f"Downloaded '{output_filename}' successfully.")

hallucination_url = 'https://drive.google.com/file/d/172SSOM4vH84Dz_SSY5OPRrtr9i1NT3o7/view?usp=drive_link'
output_filename = 'hallucination_dataset.csv'
gdown.download(hallucination_url, output_filename, quiet=False)
print(f"Downloaded '{output_filename}' successfully.")



In [ ]:
response = client.responses.create(
  model="gpt-4o-mini",
  input="Hello from ML Prague",
)

print(response.output_text)

# [INTRODUCTION](https://makkoen.github.io/logprobs-workshop/intro/)

# [RESPONSES API](https://makkoen.github.io/logprobs-workshop/intro-1/)

## Instructions

In [ ]:
INSTRUCTIONS = "Answer only in Czech"

response = client.responses.create(
  model="gpt-4o-mini",
  input=[
      {"role": "system", "content": INSTRUCTIONS},
      {"role": "user", "content": "Hello from ML Prague"},
      ],
)

print(response.output_text)

## Logprobs

In [ ]:
response = client.responses.create(
  model="gpt-4o-mini",
  input="Give me a random number. Answer only the number",
  # TODO: Setup logprobs as a part of the response object
  ...
)

print(response.output_text)

In [ ]:
print(json.dumps(response.model_dump(), indent=2))

In [ ]:
sum=0.0
for logprob in response.output[0].content[0].logprobs[0].top_logprobs:
  print(f"token: {logprob.token}, prob: {math.exp(logprob.logprob)}")
  sum += math.exp(logprob.logprob)
print(f"total: {sum}")

### Multi-token response

In [ ]:
response = client.responses.create(
  model="gpt-4o-mini",
  input="Hello from ML Prague",
  top_logprobs=2,
  include=["message.output_text.logprobs"],
)

print(response.output_text)

In [ ]:
sum=0.0
for logprob in response.output[0].content[0].logprobs[0].top_logprobs:
  print(f"token: {logprob.token}, prob: {math.exp(logprob.logprob)}")
  sum += math.exp(logprob.logprob)
print(f"total: {sum}")

In [ ]:
sum=0.0
for logprob in response.output[0].content[0].logprobs:
    print(f"token: {logprob.top_logprobs[0].token}, prob: {math.exp(logprob.logprob)}")
    sum += (logprob.logprob)

print(math.exp(sum))

In [ ]:
# Normalized
math.exp(sum/len(response.output[0].content[0].logprobs))

## Note: Reasoning Models

In [ ]:
# Not supported
response = client.responses.create(
  model="gpt-5-mini",
  input="Give me a random number. Answer only the number",
  top_logprobs=10,
  include=["message.output_text.logprobs"],
)

In [ ]:
# Without reasoning

INSTRUCTIONS = """Given a text, decide whether it's about a cat or a dog.
Output only dog or cat as your last token
"""

response = client.responses.create(
  model="gpt-4o-mini",
  input=[
      {"role": "system", "content": INSTRUCTIONS},
      {"role": "user", "content": "Hello from ML Prague"},
      ],
  top_logprobs=2,
  include=["message.output_text.logprobs"],
)

print(response.output_text)

In [ ]:
math.exp(response.output[0].content[0].logprobs[-1].top_logprobs[0].logprob)

In [ ]:
# With "reasoning"

INSTRUCTIONS = """Given a text, decide whether it's about a cat or a dog.
First think about your reasoning and then output dog or cat as your last token
"""

response = client.responses.create(
  model="gpt-4o-mini",
  input=[
      {"role": "system", "content": INSTRUCTIONS},
      {"role": "user", "content": "Hello from ML Prague"},
      ],
  top_logprobs=2,
  include=["message.output_text.logprobs"],
)

print(response.output_text)

In [ ]:
response.output[0].content[0].logprobs[-1].top_logprobs[0]

In [ ]:
math.exp(response.output[0].content[0].logprobs[-1].top_logprobs[0].logprob)


# [Binary classification](https://makkoen.github.io/logprobs-workshop/intro-2/)

## Data

In [ ]:
df = pd.read_csv("filtering_dataset.csv")
df.head(5)

## Classification using LLM

In [ ]:
INSTRUCTIONS = """Given a police/firefigther/EMS radio transcript, decide whether the transcript is about an incident happening or a just a noise.
An incident must contain:
* what is happening
* address where it's happening.
Output only 'incident' or 'noise'.
"""

response = client.responses.create(
    model="gpt-4o-mini",
    input=[
    {"role": "system", "content": INSTRUCTIONS},
    {"role": "user", "content": df.iloc[0].transcription},
    ],
    top_logprobs=2,
    include=["message.output_text.logprobs"],
)

In [ ]:
response.output[0].content[0].logprobs[0].top_logprobs[0].token

In [ ]:
math.exp(response.output[0].content[0].logprobs[0].top_logprobs[0].logprob)

In [ ]:
INSTRUCTIONS = """Given a police/firefigther/EMS radio transcript, decide whether the transcript is about an incident happening or a just a noise.
An incident must contain:
* what is happening
* address where it's happening.
Output only 'incident' or 'noise'.
"""

output_token = []
output_prob = []
# TODO: # Iterate over the dataframe a query the API
for _, row in tqdm(df.iterrows(), total=df.shape[0]):
    # TODO: # Iterate over logprobs and add probability to "incident" token
    ...

    output_prob.append(prob)

df["output_token"] = output_token
df["output_prob"] = output_prob

In [ ]:
# Calculate precision and recall
precision, recall, thresholds = precision_recall_curve(df['label'], df['output_prob'])
pr_auc = auc(recall, precision)

# Calculate baseline precision (proportion of positive class)
positive_class_ratio = df['label'].mean()

# Calculate precision and recall at 0.5 threshold
pred_threshold_0_5 = (df['output_prob'] >= 0.5).astype(int)
precision_0_5 = precision_score(df['label'], pred_threshold_0_5, zero_division=0)
recall_0_5 = recall_score(df['label'], pred_threshold_0_5, zero_division=0)

# Plot the curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'Precision-Recall curve (AUC = {pr_auc:.2f})')
plt.axhline(y=positive_class_ratio, color='r', linestyle='--', label=f'Random Classifier (Precision = {positive_class_ratio:.2f})') # Add random baseline
plt.scatter(recall_0_5, precision_0_5, color='green', marker='o', s=100, label=f'Threshold 0.5 (P: {precision_0_5:.2f}, R: {recall_0_5:.2f})', zorder=5) # Add point for 0.5 threshold
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='best')
plt.grid(True)
plt.show()

# [Hallucination Detection](https://makkoen.github.io/logprobs-workshop/intro-3/)

- Original task: Given police/firefighter/EMS radio transcription extract address and what's happening

- Output: address + title


- Our task: Decide whether given address or title is wrong/hallucinated

In [ ]:
df = pd.read_csv("hallucination_dataset.csv")

In [ ]:
df.head()

## Using LLM confidence (logprobs)


### Note: [Proper Confidence Estimation](https://makkoen.github.io/logprobs-workshop/model-confidence/)

In [ ]:
address_correct_df = df[df["address WRONG"] == 0]
address_wrong_df = df[df["address WRONG"] == 1]

plt.figure(figsize=(10, 6))
plt.hist(address_correct_df['address_prob_norm'], bins=30, alpha=0.5, label='Correct Address', density=True)
plt.hist(address_wrong_df['address_prob_norm'], bins=30, alpha=0.5, label='Wrong Address', density=True)
plt.xlabel('address_prob_norm')
plt.ylabel('Density')
plt.title('Distribution of address_prob_norm')
plt.legend(loc='upper right')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
address_correct_df = df[df["address WRONG"] == 0]
address_wrong_df = df[df["address WRONG"] == 1]

plt.figure(figsize=(10, 6))
plt.hist(address_correct_df['address_prob_abs'], bins=30, alpha=0.5, label='Correct Address', density=True)
plt.hist(address_wrong_df['address_prob_abs'], bins=30, alpha=0.5, label='Wrong Address', density=True)
plt.xlabel('address_prob_abs')
plt.ylabel('Density')
plt.title('Distribution of address_prob_abs')
plt.legend(loc='upper right')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
title_correct_df = df[df["title WRONG"] == 0]
title_wrong_df = df[df["title WRONG"] == 1]

plt.figure(figsize=(10, 6))
plt.hist(title_correct_df['title_prob_norm'], bins=30, alpha=0.5, label='Correct title', density=True)
plt.hist(title_wrong_df['title_prob_norm'], bins=30, alpha=0.5, label='Wrong title', density=True)
plt.xlabel('title_prob_norm')
plt.ylabel('Density')
plt.title('Distribution of title_prob_norm')
plt.legend(loc='upper right')
plt.grid(axis='y', alpha=0.3)
plt.show()

## Using similarity (embedding)

In [ ]:
tqdm.pandas()

def get_embedding(text, model="text-embedding-3-small"):
    if pd.isna(text) or text == "":
        return None
    text = str(text).replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding

print("Generating embeddings for 'transcription'...")
df['transcription_embedding'] = df['transcription'].progress_apply(lambda x: get_embedding(x))

print("Generating embeddings for 'address'...")
df['address_embedding'] = df['address'].progress_apply(lambda x: get_embedding(x))

print("Generating embeddings for 'title'...")
df['title_embedding'] = df['title'].progress_apply(lambda x: get_embedding(x))

display(df[['transcription_embedding', 'address_embedding', 'title_embedding']].head())

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(vec1, vec2):
    if vec1 is None or vec2 is None:
        return None
    return cosine_similarity([vec1], [vec2])[0][0]

# Calculate similarities
print("Calculating cosine similarities...")
df['similarity_transcription_address'] = df.apply(
    lambda row: calculate_cosine_similarity(row['transcription_embedding'], row['address_embedding']), axis=1
)
df['similarity_transcription_title'] = df.apply(
    lambda row: calculate_cosine_similarity(row['transcription_embedding'], row['title_embedding']), axis=1
)

display(df[['similarity_transcription_address', 'similarity_transcription_title']].head())

In [ ]:
# Re-split dataframes to include the new similarity columns
address_correct_df = df[df["address WRONG"] == 0]
address_wrong_df = df[df["address WRONG"] == 1]

title_correct_df = df[df["title WRONG"] == 0]
title_wrong_df = df[df["title WRONG"] == 1]

# Plot Address Similarity Histograms
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(address_correct_df['similarity_transcription_address'].dropna(), bins=30, alpha=0.5, label='Correct Address', density=True)
plt.hist(address_wrong_df['similarity_transcription_address'].dropna(), bins=30, alpha=0.5, label='Wrong Address', density=True)
plt.xlabel('Cosine Similarity')
plt.ylabel('Density')
plt.title('Transcription vs Address Similarity')
plt.legend()

# Plot Title Similarity Histograms
plt.subplot(1, 2, 2)
plt.hist(title_correct_df['similarity_transcription_title'].dropna(), bins=30, alpha=0.5, label='Correct Title', density=True)
plt.hist(title_wrong_df['similarity_transcription_title'].dropna(), bins=30, alpha=0.5, label='Wrong Title', density=True)
plt.xlabel('Cosine Similarity')
plt.ylabel('Density')
plt.title('Transcription vs Title Similarity')
plt.legend()

plt.tight_layout()
plt.show()

## LLM Judge

In [ ]:
INSTRUCTIONS_TITLE = """
Given a radio transcript and description, decide whether the given description is incorrect or not supported by the transcription fully.
Output only yes for incorrect description, no for correct.
"""
INSTRUCTIONS_ADDRESS = """
Given a radio transcript and address, decide whether the given address is incorrect or not supported by the transcription fully.
Output only yes for incorrect description, no for correct.
"""

output_prob_title = []
output_prob_address = []

for _, row in tqdm(df.iterrows(), total=df.shape[0]):
    # Judge Title
    res_title = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": INSTRUCTIONS_TITLE},
            {"role": "user", "content": f"Transcript: {row.transcription}\nDescription: {row.title}"}
        ],
        top_logprobs=4,
        include=["message.output_text.logprobs"]
    )

    prob_title = 0.0
    for top_lp in res_title.output[0].content[0].logprobs[0].top_logprobs:
        if top_lp.token.lower().strip().lower() == "yes":
            prob_title += math.exp(top_lp.logprob)
    output_prob_title.append(prob_title)

    # Judge Address
    res_address = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": INSTRUCTIONS_ADDRESS},
            {"role": "user", "content": f"Transcript: {row.transcription}\nAddress: {row.processed_address}"}
        ],
        top_logprobs=4,
        include=["message.output_text.logprobs"]
    )

    prob_address = 0.0
    for top_lp in res_address.output[0].content[0].logprobs[0].top_logprobs:
        if top_lp.token.lower().strip().lower() == "yes":
            prob_address += math.exp(top_lp.logprob)
    output_prob_address.append(prob_address)

df['llm_judge_title_prob'] = output_prob_title
df['llm_judge_address_prob'] = output_prob_address

In [ ]:
# Filter for Address
address_correct_df = df[df['address WRONG'] == 0]
address_wrong_df = df[df['address WRONG'] == 1]

plt.figure(figsize=(10, 5))

plt.hist(address_correct_df['llm_judge_address_prob'], bins=10, alpha=0.5, label='Correct Address', density=True)
plt.hist(address_wrong_df['llm_judge_address_prob'], bins=10, alpha=0.5, label='Wrong Address', density=True)

plt.xlabel('Judge "Yes" Probability')
plt.ylabel('Density')
plt.title('LLM Judge: Address Hallucination Prob')
plt.legend()
plt.show()

In [ ]:
# Filter for Title
title_correct_df = df[df['title WRONG'] == 0]
title_wrong_df = df[df['title WRONG'] == 1]

plt.figure(figsize=(10, 5))

plt.hist(title_correct_df['llm_judge_title_prob'], bins=10, alpha=0.5, label='Correct Title', density=True)
plt.hist(title_wrong_df['llm_judge_title_prob'], bins=10, alpha=0.5, label='Wrong Title', density=True)

plt.xlabel('Judge "Yes" Probability')
plt.ylabel('Density')
plt.title('LLM Judge: Title Hallucination Prob')
plt.legend()
plt.show()

## Putting it all together

In [ ]:
# Define global WRONG flag
df['WRONG'] = ((df['address WRONG'] == 1) | (df['title WRONG'] == 1)).astype(int)
print(f"Total wrong samples: {df['WRONG'].sum()} out of {len(df)}")

In [ ]:
# Update labels based on cleaned df
df['CORRECT'] = (df['WRONG'] == 0).astype(int)

# setting thresholds to most permissive
addr_conf_th = 0.0
title_conf_th = 0.0
addr_sim_th = 0.0
title_sim_th = 0.0
addr_judge_th = 1.0
title_judge_th = 1.0

# Model decision logic for correctness
# Since NaNs are removed, we don't strictly need fillna, but it's safe to keep
pred_correct = (
    (df['address_prob_norm'].fillna(1.0) >= addr_conf_th) &
    (df['title_prob_norm'].fillna(1.0) >= title_conf_th) &
    (df['similarity_transcription_address'].fillna(1.0) >= addr_sim_th) &
    (df['similarity_transcription_title'].fillna(1.0) >= title_sim_th) &
    (df['llm_judge_address_prob'].fillna(0.0) <= addr_judge_th) &
    (df['llm_judge_title_prob'].fillna(0.0) <= title_judge_th)
).astype(int)

# Calculate metrics for the model (Correctness)
model_precision = precision_score(df['CORRECT'], pred_correct, zero_division=0)
model_recall = recall_score(df['CORRECT'], pred_correct, zero_division=0)

# Baseline (Assuming everything is correct)
baseline_precision = df['CORRECT'].mean()
baseline_recall = 1.0

print(f"--- Baseline (Assuming everything is correct) ---")
print(f"Precision (Correctness): {baseline_precision:.2%}")
print(f"Recall (Correctness): {baseline_recall:.2%}")

print(f"\n--- Model Performance ---")
print(f"Precision: {model_precision:.2%}")
print(f"Recall: {model_recall:.2%}")

### Optuna

In [ ]:
# Set Optuna logging to warning only to remove per-trial output
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # Suggest thresholds with 0.05 steps
    addr_conf_th = trial.suggest_float('addr_conf_th', 0.0, 1.0, step=0.05)
    title_conf_th = trial.suggest_float('title_conf_th', 0.0, 1.0, step=0.05)
    addr_sim_th = trial.suggest_float('addr_sim_th', 0.0, 1.0, step=0.05)
    title_sim_th = trial.suggest_float('title_sim_th', 0.0, 1.0, step=0.05)
    addr_judge_th = trial.suggest_float('addr_judge_th', 0.0, 1.0, step=0.05)
    title_judge_th = trial.suggest_float('title_judge_th', 0.0, 1.0, step=0.05)

    # Decision logic
    pred_correct = (
        (df['address_prob_norm'] >= addr_conf_th) &
        (df['title_prob_norm'] >= title_conf_th) &
        (df['similarity_transcription_address'] >= addr_sim_th) &
        (df['similarity_transcription_title'] >= title_sim_th) &
        (df['llm_judge_address_prob'] <= addr_judge_th) &
        (df['llm_judge_title_prob'] <= title_judge_th)
    ).astype(int)

    precision = precision_score(df['CORRECT'], pred_correct, zero_division=0)
    recall = recall_score(df['CORRECT'], pred_correct, zero_division=0)

    return precision, recall

# Run Multi-objective optimization
study = optuna.create_study(directions=['maximize', 'maximize'])
study.optimize(objective, n_trials=500, show_progress_bar=True)

In [ ]:
# Plotting Pareto front
fig = vis.plot_pareto_front(study, target_names=['Precision', 'Recall'])
fig.update_layout(xaxis=dict(range=[0.8, 1.0]))
fig.show()

In [ ]:
def get_best_recall_for_precision(study, min_precision=0.8):
    # Get all trials on the Pareto front
    trials = study.best_trials

    # Filter trials that meet the precision requirement
    # In our objective: values[0] is Precision, values[1] is Recall
    valid_trials = [t for t in trials if t.values[0] >= min_precision]

    if not valid_trials:
        print(f"No trials found with precision >= {min_precision}")
        return None

    # Pick the trial with the highest recall
    best_trial = max(valid_trials, key=lambda t: t.values[1])

    print(f"--- Best Trial for Min Precision: {min_precision:.2%} ---")
    print(f"Precision: {best_trial.values[0]:.2%}")
    print(f"Recall: {best_trial.values[1]:.2%}")
    print(f"Parameters: {json.dumps(best_trial.params, indent=2)}")

    return best_trial.params

# Example usage:
best_params = get_best_recall_for_precision(study, min_precision=0.95)

### Feature Importance

In [ ]:
# Define the features used in our model
features = [
    'address_prob_norm',
    'title_prob_norm',
    'similarity_transcription_address',
    'similarity_transcription_title',
    'llm_judge_address_prob',
    'llm_judge_title_prob'
]

# Best thresholds found by Optuna
# (using the best_params variable from previous cell)

# Wrapper function for SHAP: takes a numpy array of features and returns model output
def model_predict(data_array):
    # Reconstruct logic using optimized thresholds
    addr_pass = data_array[:, 0] >= best_params['addr_conf_th']
    title_pass = data_array[:, 1] >= best_params['title_conf_th']
    addr_sim_pass = data_array[:, 2] >= best_params['addr_sim_th']
    title_sim_pass = data_array[:, 3] >= best_params['title_sim_th']
    addr_judge_pass = data_array[:, 4] <= best_params['addr_judge_th']
    title_judge_pass = data_array[:, 5] <= best_params['title_judge_th']

    return (addr_pass & title_pass & addr_sim_pass & title_sim_pass & addr_judge_pass & title_judge_pass).astype(float)

# Prepare background data and the specific data to explain
X = df[features].values

# Use KernelExplainer as our model is a non-differentiable threshold heuristic
explainer = shap.KernelExplainer(model_predict, shap.sample(X, 50))
shap_values = explainer.shap_values(X, nsamples=100)

# Plot the summary of feature importance
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X, feature_names=features)